# SMIRK on the new datasets: original method vs occlusion-aware (transfer learning / fine-tuning)

`train_occlusion.py` + `src/occlusion_trainer.py`: the encoder sees occluded faces, all targets come from the clean
faces, the visible face parts are kept consistent with the pretrained SMIRK (teacher) on the clean image.

Before running:
- Settings: **Accelerator GPU T4 x1**, **Internet on**.
- Inputs: `flame2020`, `pytorch3d-wheel`, `clean-faces` (+ `clean-faces-rafdb`), the crops (`expw-crops`,
  `affectnet_1`, `rafdb-crops`) and the occlusions (`occluded-expw`, `occluded-affectnet`, `occluded-rafdb`).
  `fan-landmarks` (FAN jaw line, as the original SMIRK). Emotion loss: EMOCA's `ResNet50` as input, or set
  `EMOCA_LICENSE_ACCEPTED = True` (cell 2) after registering at emoca.is.tue.mpg.de; without it the emotion loss is off.
- To continue an interrupted run: add the output of the previous run of this notebook as input; its
  `checkpoint_last.pt` is picked up automatically.

Set `MODE`, `METHOD` and `QUICK_TEST` in the next cell (run `original` first: it is the reference for the improvements). First run with `QUICK_TEST = True` (a few minutes), then `False`
with *Save Version → Save & Run All* (runs in the background, stops with a checkpoint before Kaggle's 12 h limit).
Keep the notebook and outputs **Private** (FLAME / SMIRK / EMOCA licenses).

In [ ]:
MODE = 'transfer'      # training regime: 'transfer' (frozen backbone, head only) or 'finetune' (whole expression encoder + translator)
METHOD = 'original'    # 'original'        : SMIRK's method unchanged, trained on the clean faces of the new datasets
                       # 'occ_augment'     : SMIRK's method + occluded inputs (clean targets), losses unchanged
                       # 'occlusion_aware' : + visibility-weighted losses, visible-region consistency, occluded cycle path
QUICK_TEST = True      # True: ~30 iterations to check that everything runs

# every method is the same trainer with some of its additions switched off
METHODS = {
    'original': ['occlusion.p_occluded=0', 'train.loss_weights.visible_consistency=0', 'occlusion.cycle_occlusion=False',
                 'occlusion.landmark_weight_occluded=1', 'occlusion.reconstruction_weight_occluded=1',
                 'train.use_base_model_for_regularization=False'],
    'occ_augment': ['occlusion.p_occluded=0.5', 'train.loss_weights.visible_consistency=0', 'occlusion.cycle_occlusion=False',
                    'occlusion.landmark_weight_occluded=1', 'occlusion.reconstruction_weight_occluded=1',
                    'train.use_base_model_for_regularization=False'],
    'occlusion_aware': [],  # the defaults of the configs
}
RUN = f'{METHOD}_{MODE}'
print('run:', RUN, METHODS[METHOD])


## 1. Code and packages

In [ ]:
import os, sys, glob, subprocess, shutil, zipfile
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPU', torch.cuda.is_available())
assert torch.cuda.is_available(), 'enable the GPU accelerator'

os.chdir('/kaggle/working')  # leave /tmp/smirk before deleting it
!rm -rf /tmp/smirk && git clone -q --depth 1 -b occlusion-aware https://github.com/THUONGVIP09/smirk.git /tmp/smirk
!cd /tmp/smirk && git log --oneline -1
!pip install -q timm==0.9.16 gdown
!python -c "import albumentations" 2>/dev/null || pip install -q albumentations

wheels = glob.glob('/kaggle/input/**/pytorch3d-*.whl', recursive=True)
assert wheels, 'add the pytorch3d-wheel dataset as input'
!pip install -q {wheels[0]}
import pytorch3d; print('pytorch3d', pytorch3d.__version__)

## 2. Assets: FLAME, SMIRK_em1, FaMoS expression templates, (EMOCA)

In [ ]:
os.makedirs('/tmp/smirk/assets/FLAME2020', exist_ok=True)
local = glob.glob('/kaggle/input/**/generic_model.pkl', recursive=True) + \
        [p for p in glob.glob('/kaggle/input/**/*.zip', recursive=True) if 'flame' in os.path.basename(p).lower()]
assert local, 'add the flame2020 dataset as input'
if local[0].endswith('.pkl'):
    shutil.copytree(os.path.dirname(local[0]), '/tmp/smirk/assets/FLAME2020', dirs_exist_ok=True)
else:
    zipfile.ZipFile(local[0]).extractall('/tmp/smirk/assets/FLAME2020')
found = glob.glob('/tmp/smirk/assets/FLAME2020/**/generic_model.pkl', recursive=True)
if os.path.dirname(found[0]) != '/tmp/smirk/assets/FLAME2020':
    for name in os.listdir(os.path.dirname(found[0])):
        shutil.move(os.path.join(os.path.dirname(found[0]), name), '/tmp/smirk/assets/FLAME2020/')

!mkdir -p /tmp/smirk/pretrained_models
!gdown -q 1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE -O /tmp/smirk/pretrained_models/SMIRK_em1.pt
!gdown -q 1wEL7KPHw2kl5DxP0UAB3h9QcQLXk7BM_ -O /tmp/expression_templates_famos.zip
!unzip -q -o /tmp/expression_templates_famos.zip -d /tmp/smirk/assets/

# EMOCA emotion network (emotion loss of the original SMIRK): a ResNet50 folder (with checkpoints/) given as input, or
# downloaded like quick_install.sh once you have registered at https://emoca.is.tue.mpg.de and accepted its license
EMOCA_LICENSE_ACCEPTED = False
emoca = [p for p in glob.glob('/kaggle/input/**/ResNet50', recursive=True) if os.path.isdir(os.path.join(p, 'checkpoints'))]
emoca_zip = [p for p in glob.glob('/kaggle/input/**/*.zip', recursive=True) if 'resnet50' in os.path.basename(p).lower()]
if emoca:
    shutil.copytree(emoca[0], '/tmp/smirk/assets/ResNet50', dirs_exist_ok=True)
elif emoca_zip:  # the dataset kept the zip
    zipfile.ZipFile(emoca_zip[0]).extractall('/tmp/smirk/assets/')
elif EMOCA_LICENSE_ACCEPTED:
    !wget -q https://download.is.tue.mpg.de/emoca/assets/EmotionRecognition/image_based_networks/ResNet50.zip -O /tmp/ResNet50.zip
    !unzip -q -o /tmp/ResNet50.zip -d /tmp/smirk/assets/
# the checkpoint that src/losses/ExpressionLoss.py loads
EMOCA_CKPT = '/tmp/smirk/assets/ResNet50/checkpoints/deca-epoch=01-val_loss_total/dataloader_idx_0=1.27607644.ckpt'
USE_EMOTION_LOSS = os.path.exists(EMOCA_CKPT)
if not USE_EMOTION_LOSS and os.path.isdir('/tmp/smirk/assets/ResNet50'):
    print('WARNING: ResNet50 found but not the expected checkpoint; files:')
    !find /tmp/smirk/assets/ResNet50 -name "*.ckpt"

print('emotion loss:', USE_EMOTION_LOSS)
!ls /tmp/smirk/assets /tmp/smirk/pretrained_models

## 3. Data folders → paths.yaml

In [ ]:
import yaml
def dataset_of(path):
    path = os.path.relpath(path, '/kaggle/input').lower()
    return next((k for k in ('expw', 'affectnet', 'rafdb') if k in path), None)

clean_dirs = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/clean_meta.csv', recursive=True))
image_dirs = {dataset_of(p): p for p in glob.glob('/kaggle/input/**/images', recursive=True) if os.path.isdir(p) and dataset_of(p)}
occluded_dirs = {dataset_of(p): os.path.dirname(p) for p in glob.glob('/kaggle/input/**/occluded_meta.csv', recursive=True) if dataset_of(p)}
image_dirs = {k: v for k, v in image_dirs.items() if k in occluded_dirs}   # train on datasets with occlusions only
for name in image_dirs:
    print(f'{name:10s} images: {image_dirs[name]}\n{"":10s} occluded: {occluded_dirs[name]}')
print('clean dirs:', clean_dirs)
assert clean_dirs and image_dirs, 'add clean-faces, the crops and the occluded datasets as input'
fan_files = {dataset_of(p): p for p in glob.glob('/kaggle/input/**/*_fan.npz', recursive=True) if dataset_of(p)}
print('FAN landmarks:', fan_files or 'none (add the fan-landmarks dataset: the original SMIRK uses the FAN jaw line)')
paths = {'data': {'clean_dirs': clean_dirs, 'image_dirs': image_dirs, 'occluded_dirs': occluded_dirs, 'fan_files': fan_files}}
yaml.safe_dump(paths, open('/kaggle/working/paths.yaml', 'w'))

LOG = f'/kaggle/working/logs/{RUN}'
previous = [p for p in glob.glob(f'/kaggle/input/**/logs/{RUN}/checkpoint_last.pt', recursive=True)]
RESUME = previous[0] if previous and not QUICK_TEST else ''
print('resume from:', RESUME or '(new run)')

## 4. Train

In [ ]:
overrides = [f'train.log_path={LOG}'] + METHODS[METHOD]
if RESUME:
    overrides.append(f'resume_checkpoint={RESUME}')
if not USE_EMOTION_LOSS:
    overrides.append('train.loss_weights.emotion_loss=0')
if QUICK_TEST:
    overrides += ['train.max_iters=30', 'train.samples_per_epoch=2000', 'train.val_batches=5', 'train.visualize_every=10',
                  'train.log_losses_every=5', 'train.num_epochs=1']
os.chdir('/tmp/smirk')
!python train_occlusion.py configs/config_occlusion_{MODE}.yaml --paths /kaggle/working/paths.yaml {' '.join(overrides)}
!ls -la {LOG}

## 5. Look at the training visualizations (first column: the occluded input the encoder saw)

In [ ]:
import matplotlib.pyplot as plt, cv2
images = sorted(glob.glob(f'{LOG}/train_images/*.jpg'), key=os.path.getmtime)
if images:
    picture = cv2.imread(images[-1])[..., ::-1]
    plt.figure(figsize=(20, 20 * picture.shape[0] / picture.shape[1])); plt.imshow(picture); plt.axis('off'); plt.show()

## 6. Quick evaluation of the trained model (300 test faces; the full evaluation is notebook 04 with this checkpoint)

In [ ]:
weights = next((f'{LOG}/{n}' for n in ['model_final.pt', 'model_last.pt'] if os.path.exists(f'{LOG}/{n}')), None)
assert weights, 'no trained weights yet'
ARGS = ' '.join([f'--clean_dir "{d}"' for d in clean_dirs] + [f'--image_dir "{k}={v}"' for k, v in image_dirs.items()])
!python -m src.evaluation.occlusion_benchmark {ARGS} --checkpoint {weights} --out_dir /kaggle/working/eval_{RUN} --limit 300 --probe_train_max 5000

## 7. Save

*Save Version → Save & Run All* (GPU T4 x1). `logs/<MODE>/` keeps `model_final.pt` (weights, same format as
SMIRK_em1.pt), `checkpoint_last.pt` (to resume) and the visualizations. Create a **Private** dataset from the output
(for example `occlusion-smirk-transfer`), then evaluate it with notebook 04 (`--checkpoint`).